In [1]:
import os
import pandas as pd
import joblib

X_train = pd.read_csv(
    "creditCardFraud/data/processed/X_train_scaled.csv"
)

y_train = pd.read_csv(
    "creditCardFraud/data/processed/y_train.csv"
).squeeze("columns")

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("\nClass distribution:")
print(y_train.value_counts())

X_train shape: (226980, 30)
y_train shape: (226980,)

Class distribution:
Class
0    226602
1       378
Name: count, dtype: int64


In [2]:
from xgboost import XGBClassifier

final_scale_pos_weight = (
    (y_train == 0).sum() /
    (y_train == 1).sum()
)

print("scale_pos_weight:", final_scale_pos_weight)

final_xgb_model = XGBClassifier(
    n_estimators = 200,
    max_depth = 6,
    learning_rate = 0.1,
    subsample = 0.8,
    colsample_bytree = 0.8,
    scale_pos_weight = final_scale_pos_weight,
    objective = "binary:logistic",
    eval_metric = "logloss",
    random_state = 42,
    n_jobs = -1
)

print("Final XGBoost model configured successfully.")

scale_pos_weight: 599.4761904761905
Final XGBoost model configured successfully.


In [3]:
final_xgb_model.fit(
    X_train,
    y_train
)

print("Final XGBoost model trained successfully.")

Final XGBoost model trained successfully.


In [5]:
from sklearn.metrics import(
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

X_test = pd.read_csv(
    "creditCardFraud/data/processed/X_test_scaled.csv"
)

y_test = pd.read_csv(
    "creditCardFraud/data/processed/y_test.csv"
).squeeze("columns")

y_prob_final = final_xgb_model.predict_proba(X_test)[:, 1]

final_threshold = 0.2325

y_pred_final = (y_prob_final >= final_threshold).astype(int)

precision_final = precision_score(y_test,y_pred_final)
recall_final = recall_score(y_test,y_pred_final)
f1_final = f1_score(y_test,y_pred_final)
roc_auc_final = roc_auc_score(y_test,y_prob_final)
pr_auc_final = average_precision_score(y_test,y_prob_final)
cm_final = confusion_matrix(y_test,y_pred_final)


print(f"Threshold : {final_threshold:.4f}")
print(f"Precision : {precision_final:.4f}")
print(f"Recall    : {recall_final:.4f}")
print(f"F1 Score  : {f1_final:.4f}")
print(f"ROC-AUC   : {roc_auc_final:.4f}")
print(f"PR-AUC    : {pr_auc_final:.4f}")

print("\nConfusion Matrix:")
print(cm_final)

Threshold : 0.2325
Precision : 0.9048
Recall    : 0.8000
F1 Score  : 0.8492
ROC-AUC   : 0.9761
PR-AUC    : 0.8248

Confusion Matrix:
[[56643     8]
 [   19    76]]
